<a href="https://colab.research.google.com/github/makkergauri/ganga-water-quality/blob/main/01_download_and_extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1 - connect Drive, install PyMuPDF, make my project folders
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pymupdf

import os, time, shutil
import fitz            # PyMuPDF (fast and robust; pdfplumber hung on the 2023 file)
import pandas as pd
import requests, urllib3
urllib3.disable_warnings()

base = "/content/drive/MyDrive/ganga_project"
local = "/content/raw_local"      # fast local copies (wiped when Colab disconnects; Drive keeps the originals)
for sub in ["raw/nwmp", "raw/interstate", "raw/ueppcb"]:
    os.makedirs(f"{base}/{sub}", exist_ok=True)
print("folders ready:", sorted(os.listdir(f"{base}/raw")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
folders ready: ['interstate', 'nwmp', 'ueppcb']


In [3]:
# Cell 2 - every file I want: (folder, file name, source, period, link)
nwmp_urls = {
    2016: "https://cpcb.gov.in/wqm/2016/Water_Quality_data_of_River_2016.pdf",
    2017: "https://cpcb.gov.in/wqm/2017/Water_Quality_Rivers_2017.pdf",
    2018: "https://cpcb.gov.in/wqm/2018/WQuality_River-Data-2018.pdf",
    2019: "https://cpcb.gov.in/wqm/2019/WQuality_River-Data-2019.pdf",
    2020: "https://cpcb.gov.in/wqm/2020/WQuality_River-Data-2020.pdf",
    2021: "https://cpcb.gov.in/wqm/2021/WQuality_River-Data-2021.pdf",
    2022: "https://cpcb.gov.in/wqm/2022/WQuality_River-Data-2022.pdf",
    2023: "https://cpcb.gov.in/wqm/2023/WQuality_River-Data-2023.pdf",
    2024: "https://cpcb.gov.in/wqm/2024/WQuality_River-Data-2024.pdf",
    2025: "https://cpcb.gov.in/wqm/2025/WQuality_River-Data-2025.pdf",
}
# yearly river data (2014 left out: it's HTML pages in several parts, a different format)
files = [("nwmp", f"nwmp_rivers_{y}.pdf", "CPCB NWMP", str(y), u) for y, u in nwmp_urls.items()]

# monthly Ganga at inter-state boundaries (the six 2019-2021 files get added here later)
files += [
    ("interstate", "interstate_2023_07-08.pdf", "CPCB interstate", "2023-07 to 08",
     "https://cpcb.nic.in/NGTMC/ganga_Interstate10.pdf"),
]

# files I downloaded in my browser and uploaded to Drive by hand (Colab was too slow or hung)
uploaded_by_hand = {"nwmp_rivers_2023.pdf", "nwmp_rivers_2024.pdf", "nwmp_rivers_2025.pdf",
                    "interstate_2023_07-08.pdf"}
print(len(files), "files listed")

11 files listed


In [4]:
# Cell 3 - download only files that aren't in Drive yet
# streamed download with a hard time limit per file, so a very slow server can't hang the cell forever
headers = {"User-Agent": "Mozilla/5.0"}   # some govt sites refuse requests without a browser-like header
MAX_SECONDS = 600                          # give up on a file after 10 minutes

for folder, name, source, period, url in files:
    path = f"{base}/raw/{folder}/{name}"
    if os.path.exists(path):
        print(f"{name:28s} already in Drive, skipped")
        continue
    t0 = time.time()
    try:
        try:
            r = requests.get(url, headers=headers, timeout=60, stream=True)
        except requests.exceptions.SSLError:
            r = requests.get(url, headers=headers, timeout=60, stream=True, verify=False)
        if r.status_code != 200:
            print(f"{name:28s} FAILED (status {r.status_code}) -> download it in the browser")
            continue
        with open(path + ".part", "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                if time.time() - t0 > MAX_SECONDS:
                    raise TimeoutError
        with open(path + ".part", "rb") as f:
            is_pdf = f.read(4) == b"%PDF"
        if is_pdf:
            os.rename(path + ".part", path)
            print(f"{name:28s} downloaded ({time.time() - t0:.0f} s)")
        else:
            os.remove(path + ".part")
            print(f"{name:28s} FAILED (not a PDF - got a web page?) -> re-copy the link")
    except Exception as e:
        if os.path.exists(path + ".part"):
            os.remove(path + ".part")
        print(f"{name:28s} FAILED ({type(e).__name__}) -> download it in the browser")

nwmp_rivers_2016.pdf         already in Drive, skipped
nwmp_rivers_2017.pdf         already in Drive, skipped
nwmp_rivers_2018.pdf         already in Drive, skipped
nwmp_rivers_2019.pdf         already in Drive, skipped
nwmp_rivers_2020.pdf         already in Drive, skipped
nwmp_rivers_2021.pdf         already in Drive, skipped
nwmp_rivers_2022.pdf         already in Drive, skipped
nwmp_rivers_2023.pdf         already in Drive, skipped
nwmp_rivers_2024.pdf         already in Drive, skipped
nwmp_rivers_2025.pdf         already in Drive, skipped
interstate_2023_07-08.pdf    already in Drive, skipped


In [5]:
# Cell 4 - copy each file to local disk, check it with PyMuPDF, save sources.csv, print titles
rows, titles = [], {}
for folder, name, source, period, url in files:
    src, dst = f"{base}/raw/{folder}/{name}", f"{local}/{folder}/{name}"
    os.makedirs(f"{local}/{folder}", exist_ok=True)
    pages = text_ok = size = None
    if not os.path.exists(src):
        how = "MISSING"
        print(f"{name:28s} MISSING")
    else:
        t0 = time.time()
        shutil.copy(src, dst)
        size = round(os.path.getsize(dst) / 1e6, 2)
        with fitz.open(dst) as doc:
            pages = doc.page_count
            first = doc[0].get_text()
        text_ok = len(first.strip()) > 100                  # little or no text = probably scanned
        titles[name] = [l for l in first.splitlines() if l.strip()][:4]
        how = "browser upload" if name in uploaded_by_hand else "Colab"
        print(f"{name:28s} {size:6.2f} MB  pages={pages:4d}  text={text_ok}  ({time.time() - t0:.0f} s)")
    rows.append({"file_name": name, "source": source, "period": period, "url": url,
                 "downloaded_by": how, "size_MB": size, "pages": pages, "has_text": text_ok})

pd.DataFrame(rows).to_csv(f"{base}/sources.csv", index=False)
print("\nsaved sources.csv\n")
for name, lines in titles.items():
    print(f"--- {name} ---")
    print("\n".join(lines) if lines else "(no text on page 1)")
    print()

nwmp_rivers_2016.pdf           3.78 MB  pages=  70  text=True  (0 s)
nwmp_rivers_2017.pdf           3.33 MB  pages=  81  text=True  (0 s)
nwmp_rivers_2018.pdf           5.12 MB  pages= 103  text=True  (0 s)
nwmp_rivers_2019.pdf           3.96 MB  pages=  66  text=True  (0 s)
nwmp_rivers_2020.pdf           5.20 MB  pages=  56  text=True  (0 s)
nwmp_rivers_2021.pdf           5.41 MB  pages= 127  text=True  (0 s)
nwmp_rivers_2022.pdf           6.22 MB  pages= 161  text=True  (0 s)
MuPDF error: syntax error: expected object number

MuPDF error: format error: non-page object in page tree

nwmp_rivers_2023.pdf          12.05 MB  pages=  81  text=False  (0 s)
nwmp_rivers_2024.pdf          15.33 MB  pages=  84  text=True  (0 s)
nwmp_rivers_2025.pdf           0.47 MB  pages=   2  text=True  (0 s)
interstate_2023_07-08.pdf      0.91 MB  pages=   9  text=True  (0 s)

saved sources.csv

--- nwmp_rivers_2016.pdf ---
TABLE - 4.1: WATER QUALITY OF RIVER BEAS - 2016 
station 
code 
Station Name 

--- 

In [6]:
# Cell 5 - which pages of each yearly file hold the Ganga table?
import re

title_re = re.compile(r"water\s+quality\s+(?:data\s+)?of\s+(?:the\s+)?river\s+([a-z][a-z .\-]*)", re.I)
good_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024]   # 2023 damaged, "2025" is Yamuna only

def as_ranges(pages):
    # turn [5, 6, 7, 10] into "5-7, 10"
    out, start, prev = [], None, None
    for p in pages:
        if start is None:
            start = prev = p
        elif p == prev + 1:
            prev = p
        else:
            out.append(f"{start}-{prev}" if start != prev else f"{start}")
            start = prev = p
    if start is not None:
        out.append(f"{start}-{prev}" if start != prev else f"{start}")
    return ", ".join(out)

ganga_pages = {}
for y in good_years:
    with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
        current, pages, n_titles, n_ganga_at = None, [], 0, 0
        for i, page in enumerate(doc):
            text = page.get_text()
            if "INDEX" in text[:300].upper():
                continue                                  # skip the index page
            m = title_re.search(text)
            if m:
                n_titles += 1
                current = m.group(1).strip(" -–(").lower()
            if current and re.match(r"ganga\b", current):
                pages.append(i + 1)                       # page numbers as shown in a PDF viewer
            n_ganga_at += len(re.findall(r"\bganga\s+at\b", text, re.I))   # \b so "Ramganga" doesn't count
        ganga_pages[y] = pages
        print(f"{y}: Ganga table on pages {as_ranges(pages) or 'NOT FOUND'} | "
              f"{n_titles} table titles in file | 'Ganga at' appears {n_ganga_at} times")

# a look at how the first Ganga page starts, for two years with different layouts
for y in [2017, 2022]:
    if ganga_pages[y]:
        with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
            lines = [l for l in doc[ganga_pages[y][0] - 1].get_text().splitlines() if l.strip()]
        print(f"\n--- {y}, page {ganga_pages[y][0]}, first 15 lines ---")
        print("\n".join(lines[:15]))

2016: Ganga table on pages 8-10 | 20 table titles in file | 'Ganga at' appears 37 times
2017: Ganga table on pages 11-16 | 20 table titles in file | 'Ganga at' appears 58 times
2018: Ganga table on pages 11-17 | 19 table titles in file | 'Ganga at' appears 51 times
2019: Ganga table on pages 12-16 | 15 table titles in file | 'Ganga at' appears 64 times
2020: Ganga table on pages 13-16 | 16 table titles in file | 'Ganga at' appears 73 times
2021: Ganga table on pages 20-27 | 20 table titles in file | 'Ganga at' appears 80 times
2022: Ganga table on pages 22-30 | 19 table titles in file | 'Ganga at' appears 71 times
2024: Ganga table on pages 9-13 | 17 table titles in file | 'Ganga at' appears 72 times

--- 2017, page 11, first 15 lines ---
Station Code LOCATION 
STATE 
Temperature 
Dissolved Oxygen 
pH 
Conductivity (µmho/ 
Cm) 
Bio-
Chemical 
Oxygen 
Demand 
(mg/L) 
Nitrate 
Faecal Coliform 
Total Coliform 

--- 2022, page 22, first 15 lines ---
Table – 3: Water Quality of River Ganga 

In [7]:
# Cell 7 - test table extraction on one year (2022) before writing a parser for all years
def short(c, n=18):
    # tidy a cell for printing: no line breaks, cut to n characters
    return (c or "").replace("\n", " ").strip()[:n]

y = 2022
first, last = ganga_pages[y][0], ganga_pages[y][-1]
with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
    for p in range(max(first - 1, 1), min(last + 1, doc.page_count) + 1):   # one page either side, to be safe
        tabs = doc[p - 1].find_tables()
        print(f"\n=== page {p}: {len(tabs.tables)} table(s) ===")
        for k, t in enumerate(tabs.tables):
            data = t.extract()
            n_ganga = sum(any("GANGA" in (c or "").upper() for c in row) for row in data)
            print(f"table {k}: {len(data)} rows x {len(data[0]) if data else 0} columns, "
                  f"{n_ganga} rows mention GANGA")
            for row in data[:5]:
                print("   ", [short(c) for c in row])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.

=== page 21: 1 table(s) ===
table 0: 20 rows x 63 columns, 0 rows mention GANGA
    ['', 'Station', '', '', 'Name Of Monitoring', '', '', 'State Name', '', '', 'Temperature', '', '', '', '', '', 'Dissolved', '', '', '', '', '', 'pH', '', '', '', '', '', 'Conductivity', '', '', '', '', '', 'Bio-Chemical', '', '', '', '', '', 'Nitrate', '', '', '', '', '', 'Fecal Coliform', '', '', '', '', '', 'Total Coliform', '', '', '', '', '', 'Fecal', '', '', '', '']
    ['', 'Code', '', '', 'Location', '', '', '', '', '', '(oC)', '', '', '', '', '', 'Oxygen', '', '', '', '', '', '', '', '', '', '', '', '(µmhos/cm)', '', '', '', '', '', 'Oxygen', '', '', '', '', '', '(mg/L)', '', '', '', '', '', '(MPN/100mL)', '', '', '', '', '', '(MPN/100mL)', '', '', '', '', '', 'Streptococci', '', '', '', '']
    ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '(mg/L)', '', '', '', '', '', '', '', '', '', '', 

In [8]:
# Cell 8 - for 2022: which column numbers hold the actual values in Ganga rows?
from collections import Counter

y = 2022
ganga_rows = []
with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
    for p in ganga_pages[y]:
        for t in doc[p - 1].find_tables().tables:
            for row in t.extract():
                if any("GANGA" in (c or "").upper() for c in row):
                    ganga_rows.append((p, row))

print(f"{len(ganga_rows)} rows mention GANGA in {y}\n")

# a few example rows: only the non-empty cells, as (column number, value)
for p, row in ganga_rows[4:8]:
    cells = [(i, short(c, 22)) for i, c in enumerate(row) if c and c.strip()]
    print(f"page {p}, {len(cells)} non-empty cells:")
    print("   ", cells, "\n")

# how many non-empty cells do Ganga rows have? (a full row should have the same number every time)
n_filled = Counter(sum(1 for c in row if c and c.strip()) for p, row in ganga_rows)
print("non-empty cells per row -> number of rows:", dict(sorted(n_filled.items())))

# which column numbers are used by Ganga rows, and how often?
used = Counter(i for p, row in ganga_rows for i, c in enumerate(row) if c and c.strip())
print("\ncolumns used (column number: rows using it):")
print(dict(sorted(used.items())))

94 rows mention GANGA in 2022

page 23, 21 non-empty cells:
    [(0, '3997'), (3, 'RIVER GANGA U/S AT ABI'), (6, 'UTTARAKHAND'), (9, '15.5'), (12, '30.6'), (15, '7.8'), (18, '10'), (21, '7.1'), (24, '8.37'), (27, '132'), (30, '664'), (33, '1.2'), (36, '2.6'), (39, '0.32'), (42, '0.9'), (45, '26'), (48, '48'), (51, '70'), (54, '110'), (57, '8'), (60, '25')] 

page 23, 21 non-empty cells:
    [(0, '10148'), (3, 'RIVER GANGA AT HAR-KI-'), (6, 'UTTARAKHAND'), (9, '14.3'), (12, '27.3'), (15, '7.8'), (18, '10.2'), (21, '7.1'), (24, '8.3'), (27, '124'), (30, '1961'), (33, '1.2'), (36, '2.4'), (39, '0.32'), (42, '1.04'), (45, '21'), (48, '48'), (51, '58'), (54, '120'), (57, '4'), (60, '26')] 

page 23, 21 non-empty cells:
    [(0, '1061'), (3, 'RIVER GANGA D/S AT BAL'), (6, 'UTTARAKHAND'), (9, '16'), (12, '29'), (15, '7.4'), (18, '10.4'), (21, '7.2'), (24, '8.6'), (27, '165'), (30, '708'), (33, '1.1'), (36, '2.8'), (39, '0.32'), (42, '1.36'), (45, '17'), (48, '84'), (51, '63'), (54, '170'), (5

In [9]:
# Cell 9 - run the 2022 checks on every year, to see whether the table layout is the same
all_rows = {}   # year -> list of (page, row) for rows mentioning GANGA; reused later

for y in good_years:
    with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
        pages = range(max(ganga_pages[y][0] - 1, 1), min(ganga_pages[y][-1] + 1, doc.page_count) + 1)
        rows_g, per_page, no_table, header = [], {}, [], None
        for p in pages:
            tabs = doc[p - 1].find_tables().tables
            if not tabs:
                no_table.append(p)
                continue
            if header is None and p in ganga_pages[y]:
                header = tabs[0].extract()[:6]          # first rows of the first Ganga table = header
            n = 0
            for t in tabs:
                for row in t.extract():
                    if any("GANGA" in (c or "").upper() for c in row):
                        rows_g.append((p, row))
                        n += 1
            per_page[p] = n
    all_rows[y] = rows_g

    n_filled = Counter(sum(1 for c in r if c and c.strip()) for p, r in rows_g)
    used = sorted({i for p, r in rows_g for i, c in enumerate(r) if c and c.strip()})
    labels = []
    if header:
        for i in range(len(header[0])):
            txt = " ".join(short(row[i], 40) for row in header if i < len(row)).strip()
            if txt:
                labels.append(f"{i}:{txt[:22]}")

    print(f"===== {y} =====")
    print(f"GANGA rows: {len(rows_g)} | per page: {per_page} | pages without a table: {no_table or 'none'}")
    print(f"non-empty cells per row -> rows: {dict(sorted(n_filled.items()))}")
    print(f"columns used: {used}")
    print(f"header: {labels}\n")

===== 2016 =====
GANGA rows: 57 | per page: {7: 0, 8: 17, 9: 31, 10: 9, 11: 0} | pages without a table: none
non-empty cells per row -> rows: {15: 5, 19: 52}
columns used: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
header: ['0:station  code', '1:Station Name', '2:State Name', '3:Temperature ⁰C   min', '4:max', '5:Dissolved oxygen  (mg/', '6:max', '7:pH   min', '8:max', '9:Conductivity  (µmhos/c', '10:max', '11:B.O.D.  (mg/l)  min', '12:max', '13:Nitrate-N + Nitrite-N ', '14:max', '15:Faecal Coli form  (MPN', '16:max', '17:Total Coli form  (MPN/', '18:max']

===== 2017 =====
GANGA rows: 71 | per page: {10: 0, 11: 0, 12: 10, 13: 14, 14: 12, 15: 16, 16: 15, 17: 4} | pages without a table: none
non-empty cells per row -> rows: {13: 7, 17: 30, 19: 34}
columns used: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
header: ['3:Temperature', '5:Dissolved Oxygen', '7:pH', '9:Conductivity (µmho/  C', '11:Bio- Chemical  Oxygen', '13:Nitrate', '15:Faecal C

In [10]:
# Cell 10 - look inside the odd rows: what got merged or wrapped?
def full(c, n=160):
    # show a whole cell, with its internal line breaks visible as " | "
    return (c or "").replace("\n", " | ").strip()[:n]

def n_cells(row):
    return sum(1 for c in row if c and c.strip())

# 1) rows with only 1 non-empty cell (2018: 4 rows, 2020: 22 rows)
for y in [2018, 2020]:
    odd = [(p, r) for p, r in all_rows[y] if n_cells(r) == 1]
    print(f"===== {y}: {len(odd)} rows with a single non-empty cell (first 5) =====")
    for p, r in odd[:5]:
        print(f"page {p}:", [(i, full(c)) for i, c in enumerate(r) if c and c.strip()])
    print()

# 2) the first rows of the tables on 2018 Ganga pages: are station rows merged into the header?
print("===== 2018: first 2 rows of each table on pages 12 and 13 =====")
with fitz.open(f"{local}/nwmp/nwmp_rivers_2018.pdf") as doc:
    for p in [12, 13]:
        for t in doc[p - 1].find_tables().tables:
            for k, row in enumerate(t.extract()[:2]):
                print(f"page {p}, row {k}:", [(i, full(c, 120)) for i, c in enumerate(row) if c and c.strip()])
        print()

# 3) one example of a short row in 2016 and 2017 (missing values = blank cells?)
for y, want in [(2016, 15), (2017, 13)]:
    ex = [(p, r) for p, r in all_rows[y] if n_cells(r) == want]
    if ex:
        p, r = ex[0]
        print(f"===== {y}: example row with {want} values (page {p}) =====")
        print([(i, short(c, 22)) for i, c in enumerate(r)])
        print()

===== 2018: 4 rows with a single non-empty cell (first 5) =====
page 14: [(2, 'GANGA BRIDGE')]
page 15: [(2, 'D/S (GANGA')]
page 16: [(2, 'GANGA)')]
page 17: [(2, 'GANGA,')]

===== 2020: 22 rows with a single non-empty cell (first 5) =====
page 13: [(1, 'GANGA RIVER AFTER CONFLUENCE OF RIVER SONG')]
page 13: [(1, 'RIVER GANGA U/S HARIDWAR AT ABINDUGHAT,')]
page 13: [(1, 'GANGA AT HARIDWAR D/S, UPPER GANGA CANAL')]
page 13: [(1, 'RIVER GANGA AT MADHYA GANGA BARRAGE,')]
page 14: [(1, 'GANGA AT KANPUR D/S (JAJMAU PUMPING')]

===== 2018: first 2 rows of each table on pages 12 and 13 =====
page 12, row 0: [(0, 'Station | Code'), (1, 'STATION NAME'), (4, 'STATE'), (7, 'Temperature | (0 C)'), (13, 'Dissolved | Oxygen | (mg/ L)'), (19, 'pH'), (25, 'Conductivity | (µmho/ Cm)'), (31, 'Bio-Chemical | Oxygen Demand | (mg/L)'), (37, 'Nitrate + | Nitrite | (mg/ L)'), (43, 'Faecal Coliform | (MPN/ 100 mL)'), (49, 'Total Coliform | (MPN/ 100 mL)')]
page 12, row 1: [(7, '_'), (13, '> 5.0 mg/L'), (19, '

In [13]:
# Cell 11 (v3) - extract every data row on the Ganga pages (±1 page), all years
# complete rows: read in order (nothing missing, so nothing can shift)
# incomplete rows: read by column position, trying every complete-row layout in the same table
code_re = re.compile(r"^\d{3,5}$")
header_word = re.compile(r"\b(code|station|location|state|name|monitoring|temperature|dissolved|oxygen|bio|"
                         r"chemical|demand|conductivity|nitrate|nitrite|coliform|faecal|fecal|total|"
                         r"streptococci|min|max|mg|ml|mpn|ph)\b", re.I)
P8 = ["temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc"]     # 2016-2018 (no fecal streptococci)
P9 = P8 + ["fs"]                                                    # 2019 onwards
params_by_year = {2016: P8, 2017: P8, 2018: P8, 2019: P9, 2020: P9, 2021: P9, 2022: P9, 2024: P9}
fixed_columns = {2016, 2017}      # these years use plain fixed columns 0, 1, 2, ...

def clean(s):
    return (s or "").replace("\n", " ").strip()

def nonempty(row):
    return [(i, clean(c)) for i, c in enumerate(row) if c and c.strip()]

def is_data(ne):
    return bool(ne) and bool(code_re.match(ne[0][1].replace(" ", "")))

def fit(ne, layout, n_fields):
    # put each cell into the nearest column of this layout; None if any cell doesn't fit
    fields = [""] * n_fields
    for i, v in ne:
        f = min(range(n_fields), key=lambda f: abs(layout[f] - i))
        if abs(layout[f] - i) > 1 or fields[f] != "":
            return None
        fields[f] = v
    return fields

records, summary = [], []
for y in good_years:
    params = params_by_year[y]
    n_fields = 3 + 2 * len(params)                    # code, location, state + min/max per parameter
    n_frag = n_flag = 0
    with fitz.open(f"{local}/nwmp/nwmp_rivers_{y}.pdf") as doc:
        pages = range(max(ganga_pages[y][0] - 1, 1), min(ganga_pages[y][-1] + 1, doc.page_count) + 1)
        for p in pages:
            for t in doc[p - 1].find_tables().tables:
                data = t.extract()
                boxes = [r.bbox for r in t.rows]
                data_rows = [(k, nonempty(row)) for k, row in enumerate(data) if is_data(nonempty(row))]
                if not data_rows:
                    continue
                layouts = Counter(tuple(i for i, _ in ne) for _, ne in data_rows if len(ne) == n_fields)
                here = []
                for k, ne in data_rows:
                    flag = ""
                    if y in fixed_columns:
                        row = data[k]
                        fields = [clean(row[f]) if f < len(row) else "" for f in range(n_fields)]
                    elif len(ne) == n_fields:
                        fields = [v for _, v in ne]                      # complete row: safe to read in order
                    else:
                        fields = None
                        for layout, _ in layouts.most_common():
                            fields = fit(ne, layout, n_fields)
                            if fields:
                                break
                        if fields is None:
                            flag = f"{len(ne)} cells, no matching layout - check by hand"
                            fields = ([v for _, v in ne] + [""] * n_fields)[:n_fields]
                    if flag:
                        n_flag += 1
                    rec = {"year": y, "source_file": f"nwmp_rivers_{y}.pdf", "page": p,
                           "station_code": fields[0].replace(" ", ""), "location_raw": fields[1],
                           "state_raw": fields[2], "name_fragments": "", "row_flag": flag,
                           "row_text": " | ".join(v for _, v in ne)}   # full raw row, for checks
                    for j, prm in enumerate(params):
                        rec[f"{prm}_min"], rec[f"{prm}_max"] = fields[3 + 2 * j], fields[4 + 2 * j]
                    here.append((k, rec))
                # wrapped name pieces: must sit in the location column and contain no header words
                if y in fixed_columns or not layouts:
                    loc_col = 1
                else:
                    loc_col = layouts.most_common(1)[0][0][1]
                for k, row in enumerate(data):
                    ne = nonempty(row)
                    if is_data(ne) or not (0 < len(ne) <= 2):
                        continue
                    if any(re.search(r"\d", v) or header_word.search(v) for _, v in ne):
                        continue
                    if abs(ne[0][0] - loc_col) > 2:
                        continue
                    text = " ".join(v for _, v in ne)
                    if len(boxes) == len(data):
                        cy = (boxes[k][1] + boxes[k][3]) / 2
                        kk, rec = min(here, key=lambda kr: abs((boxes[kr[0]][1] + boxes[kr[0]][3]) / 2 - cy))
                    else:
                        later = [kr for kr in here if kr[0] > k]
                        kk, rec = later[0] if later else here[-1]
                    rec["name_fragments"] += f"[{'before' if k < kk else 'after'}] {text} "
                    n_frag += 1
                records += [rec for _, rec in here]
    yr = [r for r in records if r["year"] == y]
    codes = [r["station_code"] for r in yr]
    summary.append({"year": y, "data_rows": len(yr), "unique_codes": len(set(codes)),
                    "duplicate_codes": len(codes) - len(set(codes)), "name_fragments": n_frag,
                    "flagged_rows": n_flag,
                    "rows_with_GANGA": sum("GANGA" in (r["row_text"] + r["name_fragments"]).upper() for r in yr)})

raw = pd.DataFrame(records)
os.makedirs(f"{base}/interim", exist_ok=True)
raw.to_csv(f"{base}/interim/nwmp_extracted_raw.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))
print(f"\nsaved {len(raw)} rows to interim/nwmp_extracted_raw.csv\n")

g = (raw["row_text"] + raw["name_fragments"]).str.upper().str.contains("GANGA")
print("still flagged (any river):", int((raw["row_flag"] != "").sum()), "| of which Ganga:", int(((raw["row_flag"] != "") & g).sum()))
for _, r in raw[(raw["row_flag"] != "") & g].head(8).iterrows():
    print(f"  {r['year']} p{r['page']} {r['station_code']}: {r['row_text'][:150]}")

print("\nGanga rows with an EMPTY location name (name only in fragments):", int((g & (raw["location_raw"] == "")).sum()))

 year  data_rows  unique_codes  duplicate_codes  name_fragments  flagged_rows  rows_with_GANGA
 2016         85            85                0               0             0               57
 2017        101           101                0               5             0               71
 2018        116           114                2             118             9               65
 2019        120           120                0              11             0               85
 2020        126           126                0              48             0               91
 2021        117           117                0               0             0               92
 2022        125           125                0               2             0               94
 2024        148           148                0              34             0              105

saved 938 rows to interim/nwmp_extracted_raw.csv

still flagged (any river): 9 | of which Ganga: 0

Ganga rows with an EMPTY location name (name